In [4]:
from gurobipy import Model, GRB, quicksum
import pandas as pd

# Load the transshipment data from the CSV file
file_path = '/Users/yashshah/Downloads/Trans_data.csv'
data = pd.read_csv(file_path)

# Create a Gurobi model
model = Model("Transshipment")

# Sets and Parameters
nodes = list(set(data['FromNode'].unique()).union(set(data['ToNode'].unique())))
supply = {'Youngstown': 7000, 'Pittsburgh': 13000}
demand = {'Albany': 3000, 'Houston': 7000, 'Tempe': 4000, 'Gary': 6000}

# Convert arcs data into a dictionary for easy lookup
arcs = {(row['FromNode'], row['ToNode']): (row['C'], row['L'], row['U']) for _, row in data.iterrows()}

# Decision variables: shipment quantities
shipment = model.addVars(arcs.keys(), lb=0, ub=GRB.INFINITY, name="shipment")

# Objective function: Minimize total shipping cost
model.setObjective(quicksum(shipment[i, j] * arcs[i, j][0] for i, j in arcs), GRB.MINIMIZE)

# Supply constraints
for node in supply:
    model.addConstr(quicksum(shipment[node, j] for j in nodes if (node, j) in arcs) <= supply[node], f"supply_{node}")

# Demand constraints
for node in demand:
    model.addConstr(quicksum(shipment[i, node] for i in nodes if (i, node) in arcs) == demand[node], f"demand_{node}")

# Flow conservation constraints for transshipment nodes (Cincinnati, Kansas City, Chicago)
transshipment_nodes = {'Cincinnati', 'Kansas City', 'Chicago'}
for node in transshipment_nodes:
    inflow = quicksum(shipment[i, node] for i in nodes if (i, node) in arcs)
    outflow = quicksum(shipment[node, j] for j in nodes if (node, j) in arcs)
    model.addConstr(inflow == outflow, f"flow_{node}")

# Lower and upper bounds on shipment quantities
for (i, j) in arcs:
    model.addConstr(shipment[i, j] >= arcs[i, j][1], f"min_shipment_{i}_{j}")
    model.addConstr(shipment[i, j] <= arcs[i, j][2], f"max_shipment_{i}_{j}")

# Optimize the model
model.optimize()

# Extract solution
if model.status == GRB.OPTIMAL:
    solution = {(i, j): shipment[i, j].x for (i, j) in arcs if shipment[i, j].x > 0}
    solution_df = pd.DataFrame([(i, j, v) for (i, j), v in solution.items()], columns=['From', 'To', 'Shipment'])
    total_cost = model.objVal
    solution_df['Total Cost'] = total_cost
    print("Optimal Transshipment Plan:")
    print(solution_df)
else:
    print("No optimal solution found.")


Set parameter Username
Set parameter LicenseID to value 2625267
Academic license - for non-commercial use only - expires 2026-02-19
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 23.5.0 23F79)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 37 rows, 14 columns and 52 nonzeros
Model fingerprint: 0xbb1d3364
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+03, 1e+04]
Presolve removed 37 rows and 14 columns
Presolve time: 0.02s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.2980000e+07   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Optimal objective  1.298000000e+07
Optimal Transshipment Plan:
          From          To  Shipment  Total Cost
0   Youngstown      Albany    1000.0  1